# 01 Feature Extraction

This notebook loads an audio file, extracts frame-level descriptors, and writes a feature table that can be reused by the browser viewer or later notebooks.

It is designed for local analysis on your PC, so you can do the heavier work here instead of inside the browser.

In [ ]:
import json
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = False

## Configure Inputs
Set the audio file you want to analyze. If your source is a video file, extract its audio track first with ffmpeg or another local tool, then point the notebook at that audio file.

In [ ]:
AUDIO_PATH = Path('data/example.wav')
OUTPUT_DIR = Path('analysis_output')
OUTPUT_DIR.mkdir(exist_ok=True)

FRAME_SIZE = 2048
HOP_LENGTH = 1024
N_MFCC = 40

print(f'Audio path: {AUDIO_PATH.resolve()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')

In [ ]:
def load_audio(audio_path, sr=None):
    y, sample_rate = librosa.load(audio_path, sr=sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sample_rate)
    return y, sample_rate, duration

def extract_frame_features(y, sr, frame_size=2048, hop_length=1024, n_mfcc=40):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=frame_size, hop_length=hop_length)
    rms = librosa.feature.rms(y=y, frame_length=frame_size, hop_length=hop_length)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=frame_size, hop_length=hop_length)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=frame_size, hop_length=hop_length)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_size, hop_length=hop_length)[0]

    frame_count = min(len(rms), mfcc.shape[1], len(centroid), len(flatness), len(rolloff), len(bandwidth), len(zcr))
    time = librosa.frames_to_time(np.arange(frame_count), sr=sr, hop_length=hop_length)

    rows = []
    for index in range(frame_count):
        row = {
            'frame_index': index,
            'time': float(time[index]),
            'rms': float(rms[index]),
            'centroid': float(centroid[index]),
            'flatness': float(flatness[index]),
            'rolloff': float(rolloff[index]),
            'bandwidth': float(bandwidth[index]),
            'zcr': float(zcr[index]),
        }

        for coeff in range(n_mfcc):
            row[f'mfcc_{coeff + 1}'] = float(mfcc[coeff, index])

        rows.append(row)

    return pd.DataFrame(rows)

def analyze_audio(audio_path):
    y, sr, duration = load_audio(audio_path)
    features = extract_frame_features(y, sr, frame_size=FRAME_SIZE, hop_length=HOP_LENGTH, n_mfcc=N_MFCC)
    features.attrs['sample_rate'] = sr
    features.attrs['duration'] = duration
    features.attrs['audio_path'] = str(Path(audio_path).resolve())
    return y, sr, duration, features

In [ ]:
if AUDIO_PATH.exists():
    y, sr, duration, features_df = analyze_audio(AUDIO_PATH)
    display(features_df.head())
    print(f'Duration: {duration:.2f}s')
    print(f'Sample rate: {sr} Hz')
    print(f'Frames: {len(features_df)}')
else:
    print('Update AUDIO_PATH to point at a real audio file before running the analysis cells.')

In [ ]:
def save_feature_table(features_df, output_dir=OUTPUT_DIR, stem='features'):
    csv_path = output_dir / f'{stem}.csv'
    json_path = output_dir / f'{stem}.json'
    features_df.to_csv(csv_path, index=False)
    features_df.to_json(json_path, orient='records', indent=2)
    return csv_path, json_path

if 'features_df' in globals():
    csv_path, json_path = save_feature_table(features_df)
    print(f'Saved: {csv_path}')
    print(f'Saved: {json_path}')